In [26]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import sys
sys.path.append("../")

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('seaborn-v0_8-dark')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (16, 9),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import QuantLib as ql
import rateslib as rl

import datetime
import pytz
NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
UTC_tz = pytz.timezone("UTC")

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [27]:
cache_path = r"C:\Users\chris\clee\project-oasis\private\sdranalytics\.cache"
start = NY_tz.localize(datetime.datetime(2026, 1, 8, 0, 0))
end = NY_tz.localize(datetime.datetime(2026, 1, 8, 23, 59))

from SDRUtils.data.builder import SDRDataBuilder
sdr = SDRDataBuilder(cache_path=cache_path, show_tqdm=True)
df = sdr.grab_sdr_trades(
	start_timestamp=start,
	end_timestamp=end,
	agency="CFTC",
	asset_class="RATES",
)
# df

MERGING SLICES...: 100%|██████████| 2/2 [00:00<00:00, 154.56it/s]


In [28]:
# from SDRUtils.products.usd.sofr_swaps import USD_SOFR_SwapProduct 
# USD_SOFR_SwapProduct().build_classification_dataframe(start=start, end=end, cache_path=cache_path)

from SDRUtils.products.usd.usd_swaptions import USD_Swaptions, straddle_pricer_from_row
sdf = USD_Swaptions().build_classification_dataframe(start=start, end=end, cache_path=cache_path)
# sdf.head(50)

Classifying Trades: 100%|██████████| 535/535 [00:00<00:00, 1296.35trade/s]


In [23]:
# sdf["product_type"].value_counts()
sdf[(sdf["package_type"] == "STRADDLE") & ((sdf["forward_label"] == "1Y")) & ((sdf["tenor_label"] == "10Y"))]

,event_action,trade_id,execution_timestamp,effective_date,expiration_date,product_type,trade_label,notional,notional_currency,is_notional_capped,...,upi_underlier_name,unique_product_identifier,platform_identifier,cleared,package_indicator,package_transaction_price,option_premium_amount,package_confidence,package_reason,package_legs_count
173,NEWT-TRAD,1667144861000000701 / 1667144862000000801,2026-01-08 14:23:07+00:00,2026-01-08,2027-01-08,SWAPTION_RECEIVER / SWAPTION_PAYER,USD-SOFR-OIS Compound 1D CONSTANT 1Y10Y RECEIV...,45000000.0,USD,False,...,NA/Swap OIS USD,QZNLQ8T0N0SX / QZZGWPNBF5R3,BILT,N,True,"2,184,750",0,1.0,platform=; time_delta_max=0.0s; premium_mode=S...,2
225,MODI-TRAD,1668826560000000501 / 1668834089000000201,2026-01-08T17:56:30+00:00 / 2026-01-08T17:56:4...,2026-01-08,2027-01-08,SWAPTION_PAYER / SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1D CONSTANT 1Y10Y PAYER ...,60000000.0,USD,False,...,NA/Swap OIS USD,QZZGWPNBF5R3 / QZNLQ8T0N0SX,XXXX,N,False,NaN,"1,458,000",0.9,platform=; time_delta_max=11.0s; premium_mode=...,2
263,NEWT-TRAD / MODI-TRAD,1669641772000000501 / 1670753198000000101,2026-01-08 20:41:31+00:00,2026-01-08,2027-01-08,SWAPTION_RECEIVER / SWAPTION_PAYER,USD-SOFR-OIS Compound 1D CONSTANT 1Y10Y RECEIV...,100000000.0,USD,False,...,NA/Swap Fxd Flt USD,QZMMWR8JKZQ8 / QZWXKVHB5F8V,BGCD,N,True,"4,900,000","4,900,000",1.0,platform=; time_delta_max=0.0s; premium_mode=S...,2
265,MODI-TRAD,1669654417000000801 / 1670062389000000201,2026-01-08 20:43:11+00:00,2026-01-08,2027-01-08,SWAPTION_PAYER / SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1D CONSTANT 1Y10Y PAYER ...,50000000.0,USD,False,...,NA/Swap Fxd Flt USD,QZWXKVHB5F8V / QZMMWR8JKZQ8,BGCD,N,True,"2,450,000","2,450,000",1.0,platform=; time_delta_max=0.0s; premium_mode=S...,2
268,NEWT-TRAD / MODI-TRAD,1669756673000000201 / 1670010317000000201,2026-01-08 20:56:03+00:00,2026-01-08,2027-01-08,SWAPTION_RECEIVER / SWAPTION_PAYER,USD-SOFR-OIS Compound 1D CONSTANT 1Y10Y RECEIV...,50000000.0,USD,False,...,NA/Swap Fxd Flt USD,QZMMWR8JKZQ8 / QZWXKVHB5F8V,BGCD,N,True,"2,445,000","2,445,000",1.0,platform=; time_delta_max=0.0s; premium_mode=S...,2


In [24]:
sdf[(sdf["package_type"] == "STRADDLE") & ((sdf["forward_label"] == "1Y")) & ((sdf["tenor_label"] == "10Y"))].iloc[0].to_dict()

{'event_action': 'NEWT-TRAD',
 'trade_id': '1667144861000000701 / 1667144862000000801',
 'execution_timestamp': Timestamp('2026-01-08 14:23:07+0000', tz='UTC'),
 'effective_date': Timestamp('2026-01-08 00:00:00'),
 'expiration_date': Timestamp('2027-01-08 00:00:00'),
 'product_type': 'SWAPTION_RECEIVER / SWAPTION_PAYER',
 'trade_label': 'USD-SOFR-OIS Compound 1D CONSTANT 1Y10Y RECEIVER EURO VANILLA PHYS / USD-SOFR-OIS Compound 1D CONSTANT 1Y10Y PAYER EURO VANILLA PHYS',
 'notional': 45000000.0,
 'notional_currency': 'USD',
 'is_notional_capped': False,
 'estimated_pv01': 0.0,
 'package_type': 'STRADDLE',
 'package_id': 'STRADDLE_071bc2752441',
 'package_legs': ['1667144862000000801', '1667144861000000701'],
 'underlying_expiration_date': Timestamp('2037-01-12 00:00:00'),
 'tenor_years': 10.158333333333333,
 'tenor_label': '10Y',
 'forward_start_years': 1.0138888888888888,
 'forward_label': '1Y',
 'premium': 2184750.0,
 'exercise_style': 'EUROPEAN',
 'strike': 0.03915,
 'upi_underlier_n

In [17]:
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP

mdp = IRSwapsMDP(source="ERIS_EOD_LIVE-QL_BASIC")
pricer = mdp.get_pricer(request=dict(curve_name="USD-SOFR-1D", timestamp=start.date()))
pricer

QLIRSwapCurve(_ql_curve_id='USD-SOFR-1D', _ql_curve_handle=<QuantLib.QuantLib.YieldTermStructureHandle; proxy of <Swig Object of type 'Handle< YieldTermStructure > *' at 0x000001F72DE76700> >, _ql_curve_index=<QuantLib.QuantLib.Sofr; proxy of <Swig Object of type 'ext::shared_ptr< Sofr > *' at 0x000001F7301CEBE0> >, _meta_data={'timestamp': datetime.datetime(2026, 1, 8, 0, 0)})

In [30]:
# from Query.IRSwaps.IRSwapQuery import IRSwapQuery

# q = IRSwapQuery(curve="USD-SOFR-1D", effective_date=datetime.date(2027, 1, 8), maturity_date=datetime.date(2037, 1, 12), structure_kwargs={"notional": 50000000})
# pkg, rws = q.resolve_package(pricer_or_curve=pricer) 

# underlying: ql.OvernightIndexedSwap = pkg[0]

# underlying_swap_pricing_engine = ql.DiscountingSwapEngine(pricer.handle())
# underlying.setPricingEngine(underlying_swap_pricing_engine)

# observed_ql_swaption_pricing_engine = ql.BachelierSwaptionEngine(pricer.handle(), ql.QuoteHandle(ql.SimpleQuote(0.0)), pricer.daycounter())
# observed_ql_swaption = ql.Swaption(underlying, ql.EuropeanExercise(underlying.startDate()))
# observed_ql_swaption.setPricingEngine(observed_ql_swaption_pricing_engine)

straddle_pricer_from_row(sdf[(sdf["package_type"] == "STRADDLE") & ((sdf["forward_label"] == "1Y")) & ((sdf["tenor_label"] == "10Y"))].iloc[0], pricer)

(<QuantLib.QuantLib.Swaption; proxy of <Swig Object of type 'ext::shared_ptr< Swaption > *' at 0x000001F72E5C7F30> >,
 73.04674344127507)

In [19]:
observed_ql_swaption.impliedVolatility(
	price=2450000,
	discountCurve=pricer.handle(),
	guess=0.01,
	accuracy=1e-5,
	maxEvaluations=1000,
	minVol=0,
	maxVol=0.1,
	type=ql.Normal,
	displacement=0,
	priceType=ql.Swaption.Forward
) / 2 * 10_000

73.72379873578738

In [10]:
df[df["Dissemination Identifier"] == 1570700287000000601].iloc[-1].to_dict()

{'Dissemination Identifier': 1570700287000000601,
 'Original Dissemination Identifier': nan,
 'Action type': 'NEWT',
 'Event type': 'TRAD',
 'Event timestamp': Timestamp('2025-12-29 13:22:06+0000', tz='UTC'),
 'Amendment indicator': None,
 'Asset Class': 'IR',
 'Product name': None,
 'Cleared': 'N',
 'Mandatory clearing indicator': False,
 'Execution Timestamp': Timestamp('2025-12-29 13:11:18+0000', tz='UTC'),
 'Effective Date': Timestamp('2025-12-29 00:00:00'),
 'Expiration Date': Timestamp('2026-08-19 00:00:00'),
 'Maturity date of the underlier': datetime.date(2036, 8, 21),
 'Non-standardized term indicator': False,
 'Platform identifier': 'BILT',
 'Prime brokerage transaction indicator': False,
 'Block trade election indicator': False,
 'Large notional off-facility swap election indicator': False,
 'Notional amount-Leg 1': '45,000,000',
 'Notional amount-Leg 2': '45,000,000',
 'Notional currency-Leg 1': 'USD',
 'Notional currency-Leg 2': 'USD',
 'Notional quantity-Leg 1': None,
 'N